In [18]:
# Get the sqlite database path
#| echo: false

from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine

import nhs_waiting_lists as nhs
from nhs_waiting_lists import (
    __app_name__,
)
from nhs_waiting_lists.constants import LARGE_ACUTE_PROVIDER_CODES, TOTAL_ONLY_TREATMENT_CODES
from nhs_waiting_lists.constants import proj_db_path, DB_FILE
from nhs_waiting_lists.utils.xdg import XDGBasedir

project_root = Path(XDGBasedir.get_data_dir(__app_name__))

DB_PATH = project_root / proj_db_path / DB_FILE
FILES_DIR = project_root / "files"

engine = create_engine(f"sqlite:///{DB_PATH}")

# Get the 23 large acute trust provider codes, identified by the ranking table csv
PROVIDER_CODES = LARGE_ACUTE_PROVIDER_CODES

# Get the C_999 meta-treatment code which aggregates all other treatment codes
TREATMENT_CODES = TOTAL_ONLY_TREATMENT_CODES
# TREATMENT_CODES = ALL_TREATMENT_CODES


In [19]:
#| echo: false
#| output: false

## Retrieve all the provider rtt summary data

start_period = "2024-01"
end_period = "2025-09"

consolidated_df = nhs.get_consolidated_df(
    start_period,
    end_period,
    PROVIDER_CODES,
    TREATMENT_CODES,
)

consolidated_df.head()


,period,provider,treatment,untreated,new_periods,incomplete,incomplete_prev,incomplete_diff,completed,treated,provider_name,provider_type,provider_subtype,total_treatable,incomplete_expected
0,2024-01-01,R0B,C_999,-2871,18812,60893,60550,343,15598,15598,South Tyneside and Sunderland NHS Foundation T...,Acute trust,Acute - Large,79362,63764
1,2024-02-01,R0B,C_999,-2593,18924,62414,60893,1521,14810,14810,South Tyneside and Sunderland NHS Foundation T...,Acute trust,Acute - Large,79817,65007
2,2024-03-01,R0B,C_999,-3423,17715,62324,62414,-90,14382,14382,South Tyneside and Sunderland NHS Foundation T...,Acute trust,Acute - Large,80129,65747
3,2024-04-01,R0B,C_999,-5070,18755,60982,62324,-1342,15027,15027,South Tyneside and Sunderland NHS Foundation T...,Acute trust,Acute - Large,81079,66052
4,2024-05-01,R0B,C_999,-3010,19161,61709,60982,727,15424,15424,South Tyneside and Sunderland NHS Foundation T...,Acute trust,Acute - Large,80143,64719


In [20]:
# Echo the base df before applying style sheets
#| echo: false
#| output: false

aggregation_functions = {
    'incomplete': 'sum',
    'incomplete_prev': 'sum',
    'new_periods': 'sum',
    'treated': 'sum',
    'untreated': 'sum',
    'total_treatable': 'mean'
}

other_result: pd.DataFrame = consolidated_df.groupby(
    ['period', 'provider']
)[
    [
        'incomplete_prev',
        'new_periods',
        'treated',
        'untreated',
        "incomplete",
        'total_treatable'
    ]
].agg(
    aggregation_functions
).reset_index()

other_result

,period,provider,incomplete,incomplete_prev,new_periods,treated,untreated,total_treatable
0,2024-01-01,R0B,60893,60550,18812,15598,-2871,79362.0
1,2024-01-01,RAJ,160406,160359,44573,25242,-19284,204932.0
2,2024-01-01,RDE,87455,86562,20025,16595,-2537,106587.0
3,2024-01-01,RDU,80905,80256,15800,13291,-1860,96056.0
4,2024-01-01,REF,43406,44445,14120,12768,-2391,58565.0
...,...,...,...,...,...,...,...,...
478,2025-09-01,RWP,57391,56632,13378,11725,-894,70010.0
479,2025-09-01,RWY,37047,36738,10648,10776,437,47386.0
480,2025-09-01,RXC,61439,62249,15060,13077,-2793,77309.0
481,2025-09-01,RXK,66524,65604,17257,16647,310,82861.0


In [21]:

other_result["total_pathways"] = other_result["incomplete_prev"] + other_result["new_periods"]

other_result["unexplained"] = other_result["incomplete"] - (
        other_result["incomplete_prev"] + other_result["new_periods"] - other_result["treated"])

other_result["unexplained_pct"] = other_result["untreated"] / other_result["total_pathways"]

other_result["expected_pathways"] = other_result["incomplete_prev"] + other_result["new_periods"] - other_result[
    "treated"]

## Unexplained pct

In [22]:

df_sorted = other_result.sort_values(by=['unexplained_pct'])[["period", "provider", "unexplained_pct", "unexplained"]]
df_top_20 = df_sorted.head(20)
df_final = df_top_20.reset_index(drop=True)
df_final

,period,provider,unexplained_pct,unexplained
0,2024-04-01,RHW,-0.160402,-6998
1,2024-01-01,RHW,-0.157235,-6899
2,2024-10-01,RHW,-0.153375,-6941
3,2025-06-01,RHW,-0.152587,-7364
4,2025-03-01,RHW,-0.148117,-7197
5,2024-07-01,RHW,-0.141166,-6095
6,2025-09-01,RHW,-0.136855,-6134
7,2025-02-01,RHW,-0.134205,-6439
8,2025-07-01,RHW,-0.129244,-6133
9,2025-05-01,RTE,-0.128286,-11030


## Unexplained absolute top 20

In [23]:
other_result.sort_values(by=['unexplained'])[["period", "provider", "unexplained_pct", "unexplained"]].head(
    20).reset_index(drop=True)

,period,provider,unexplained_pct,unexplained
0,2024-07-01,RAJ,-0.106382,-22639
1,2024-03-01,RAJ,-0.101640,-21245
2,2024-10-01,RAJ,-0.094658,-19801
3,2024-01-01,RAJ,-0.094100,-19284
4,2024-11-01,RAJ,-0.092199,-19120
5,2024-09-01,RAJ,-0.087315,-18304
6,2024-05-01,RAJ,-0.085040,-17804
7,2024-12-01,RAJ,-0.084992,-17062
8,2025-02-01,RAJ,-0.078409,-16327
9,2024-06-01,RAJ,-0.077418,-16160


In [24]:
consolidated_df

,period,provider,treatment,untreated,new_periods,incomplete,incomplete_prev,incomplete_diff,completed,treated,provider_name,provider_type,provider_subtype,total_treatable,incomplete_expected
0,2024-01-01,R0B,C_999,-2871,18812,60893,60550,343,15598,15598,South Tyneside and Sunderland NHS Foundation T...,Acute trust,Acute - Large,79362,63764
1,2024-02-01,R0B,C_999,-2593,18924,62414,60893,1521,14810,14810,South Tyneside and Sunderland NHS Foundation T...,Acute trust,Acute - Large,79817,65007
2,2024-03-01,R0B,C_999,-3423,17715,62324,62414,-90,14382,14382,South Tyneside and Sunderland NHS Foundation T...,Acute trust,Acute - Large,80129,65747
3,2024-04-01,R0B,C_999,-5070,18755,60982,62324,-1342,15027,15027,South Tyneside and Sunderland NHS Foundation T...,Acute trust,Acute - Large,81079,66052
4,2024-05-01,R0B,C_999,-3010,19161,61709,60982,727,15424,15424,South Tyneside and Sunderland NHS Foundation T...,Acute trust,Acute - Large,80143,64719
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
478,2025-05-01,RXR,C_999,-4439,11771,57660,60237,-2577,9909,9909,East Lancashire Hospitals NHS Trust,Acute trust,Acute - Large,72008,62099
479,2025-06-01,RXR,C_999,-2562,13750,58192,57660,532,10656,10656,East Lancashire Hospitals NHS Trust,Acute trust,Acute - Large,71410,60754
480,2025-07-01,RXR,C_999,-4499,14741,57371,58192,-821,11063,11063,East Lancashire Hospitals NHS Trust,Acute trust,Acute - Large,72933,61870
481,2025-08-01,RXR,C_999,-2677,12690,57217,57371,-154,10167,10167,East Lancashire Hospitals NHS Trust,Acute trust,Acute - Large,70061,59894


In [25]:

aggregation_functions = {
    'incomplete': 'sum',
    'incomplete_prev': 'sum',
    'new_periods': 'sum',
    'treated': 'sum',
    'untreated': 'sum',
    'total_treatable': 'mean'
}

all_provider_rankings: pd.DataFrame = consolidated_df.groupby([
    'provider',
    'provider_name',
])[
    [
        "incomplete",
        'incomplete_prev',
        'new_periods',
        'total_treatable',
        'treated',
        'untreated',
    ]
].agg(
    aggregation_functions
).reset_index()

all_provider_rankings

,provider,provider_name,incomplete,incomplete_prev,new_periods,treated,untreated,total_treatable
0,R0B,South Tyneside and Sunderland NHS Foundation T...,1258335,1260163,398045,341737,-58136,78962.285714
1,RAJ,Mid and South Essex NHS Foundation Trust,3535754,3517214,819266,506492,-294234,206499.047619
2,RDE,East Suffolk and North Essex NHS Foundation Trust,1898122,1891634,403490,332984,-64018,109291.619048
3,RDU,Frimley Health NHS Foundation Trust,1673379,1683401,314135,304395,-19762,95120.761905
4,REF,Royal Cornwall Hospitals NHS Trust,882602,886552,291707,244996,-50661,56107.571429
5,RGN,North West Anglia NHS Foundation Trust,1656175,1660240,310538,295744,-18859,93846.571429
6,RH8,Royal Devon University Healthcare NHS Foundati...,1536205,1538011,370936,340875,-31867,90902.238095
7,RHU,Portsmouth Hospitals University National Healt...,1283687,1284049,284624,246052,-38934,74698.714286
8,RHW,Royal Berkshire NHS Foundation Trust,733495,730921,211783,91313,-117896,44890.666667
9,RJ2,Lewisham and Greenwich NHS Trust,1330394,1344873,281981,219102,-77358,77469.238095


In [26]:

all_provider_rankings["total_pathways"] = all_provider_rankings["incomplete_prev"] + all_provider_rankings[
    "new_periods"]

all_provider_rankings["unexplained"] = all_provider_rankings["incomplete"] - (
        all_provider_rankings["incomplete_prev"] + all_provider_rankings["new_periods"] - all_provider_rankings[
    "treated"])

all_provider_rankings["unexplained_pct"] = all_provider_rankings["untreated"] / all_provider_rankings["total_pathways"]

all_provider_rankings["expected_pathways"] = all_provider_rankings["incomplete_prev"] + all_provider_rankings[
    "new_periods"] - all_provider_rankings[
                                                 "treated"]

all_provider_rankings["actual_pathways"] = all_provider_rankings["incomplete"]

all_provider_rankings.sort_values(by=['unexplained'])[
    ["provider", 'provider_name', "unexplained_pct", "unexplained"]].head(10).reset_index(drop=True)

,provider,provider_name,unexplained_pct,unexplained
0,RAJ,Mid and South Essex NHS Foundation Trust,-0.067851,-294234
1,RTE,Gloucestershire Hospitals NHS Foundation Trust,-0.084949,-158335
2,RHW,Royal Berkshire NHS Foundation Trust,-0.125062,-117896
3,RJ2,Lewisham and Greenwich NHS Trust,-0.047551,-77358
4,RXR,East Lancashire Hospitals NHS Trust,-0.042689,-71388
5,RDE,East Suffolk and North Essex NHS Foundation Trust,-0.027893,-64018
6,R0B,South Tyneside and Sunderland NHS Foundation T...,-0.035060,-58136
7,RWH,East and North Hertfordshire NHS Trust,-0.039133,-56612
8,RTF,Northumbria Healthcare NHS Foundation Trust,-0.056589,-52427
9,REF,Royal Cornwall Hospitals NHS Trust,-0.042996,-50661


In [27]:
all_provider_rankings.sort_values(by=['unexplained_pct'])[
    ["provider", 'provider_name', "unexplained_pct", "unexplained"]].head(10).reset_index(drop=True)

,provider,provider_name,unexplained_pct,unexplained
0,RHW,Royal Berkshire NHS Foundation Trust,-0.125062,-117896
1,RTE,Gloucestershire Hospitals NHS Foundation Trust,-0.084949,-158335
2,RAJ,Mid and South Essex NHS Foundation Trust,-0.067851,-294234
3,RTF,Northumbria Healthcare NHS Foundation Trust,-0.056589,-52427
4,RJ2,Lewisham and Greenwich NHS Trust,-0.047551,-77358
5,REF,Royal Cornwall Hospitals NHS Trust,-0.042996,-50661
6,RXR,East Lancashire Hospitals NHS Trust,-0.042689,-71388
7,RWH,East and North Hertfordshire NHS Trust,-0.039133,-56612
8,R0B,South Tyneside and Sunderland NHS Foundation T...,-0.035060,-58136
9,RVJ,North Bristol NHS Trust,-0.032022,-38721


In [28]:
all_provider_rankings.sort_values(by=['unexplained_pct'], ascending=False)[
    ["provider", 'provider_name', "unexplained_pct", "unexplained", "total_treatable"]].head(5).reset_index(drop=True)

,provider,provider_name,unexplained_pct,unexplained,total_treatable
0,RXK,Sandwell and West Birmingham Hospitals NHS Trust,-0.001426,-2661,88840.333333
1,RN5,Hampshire Hospitals NHS Foundation Trust,-0.004421,-6209,66872.476190
2,RWF,Maidstone and Tunbridge Wells NHS Trust,-0.008101,-9704,57039.857143
3,RWY,Calderdale and Huddersfield NHS Foundation Trust,-0.009214,-8839,45680.523810
4,RGN,North West Anglia NHS Foundation Trust,-0.009569,-18859,93846.571429


In [29]:
(df_final.style.format(precision=2, thousands=",", decimal=".")
 .format('{:.2%}', subset=["unexplained_pct"])
 .format(lambda v: v.strftime("%Y-%m"), subset=["period"])
 )

,period,provider,unexplained_pct,unexplained
0,2024-04,RHW,-16.04%,"-6,998"
1,2024-01,RHW,-15.72%,"-6,899"
2,2024-10,RHW,-15.34%,"-6,941"
3,2025-06,RHW,-15.26%,"-7,364"
4,2025-03,RHW,-14.81%,"-7,197"
5,2024-07,RHW,-14.12%,"-6,095"
6,2025-09,RHW,-13.69%,"-6,134"
7,2025-02,RHW,-13.42%,"-6,439"
8,2025-07,RHW,-12.92%,"-6,133"
9,2025-05,RTE,-12.83%,"-11,030"


In [30]:
#| echo: false
#| output: true

(other_result[
     [
         "period",
         "incomplete",
         "expected_pathways",
         "unexplained",
         "total_pathways",
         "unexplained_pct",
     ]].head(20)
 .style.relabel_index(
    [
        "",
        'Observed<br>Incomplete<br>Pathways',
        'Expected<br>Incomplete<br>Pathways',
        'Unreported<br>Change',
        'Treatable<br>Pathways',
        'Unreported<br>%'
    ], axis=1).hide(axis='index')
 .format(precision=3, thousands=",", decimal=".")
 .format('{:.2%}', subset=["unexplained_pct"])
 .format(lambda v: v.strftime("%Y-%m"), subset=["period"])
 )

,ObservedIncompletePathways,ExpectedIncompletePathways,UnreportedChange,TreatablePathways,Unreported%
2024-01,"60,893","63,764","-2,871","79,362",-3.62%
2024-01,"160,406","179,690","-19,284","204,932",-9.41%
2024-01,"87,455","89,992","-2,537","106,587",-2.38%
2024-01,"80,905","82,765","-1,860","96,056",-1.94%
2024-01,"43,406","45,797","-2,391","58,565",-4.08%
2024-01,"81,599","83,119","-1,520","97,320",-1.56%
2024-01,"74,539","76,168","-1,629","92,556",-1.76%
2024-01,"59,369","61,380","-2,011","73,543",-2.73%
2024-01,"32,034","38,933","-6,899","43,877",-15.72%
2024-01,"67,679","72,087","-4,408","83,362",-5.29%
